# Visualizaciones: Educación, Desigualdad y Planes Sociales

Este notebook genera tres gráficos interactivos construidos con Altair:
1. **Trayectoria: Inversión Educativa vs. Protección Social** — Argentina (2006–2022)
2. **Relación entre inversión educativa y desigualdad** — Argentina, Brasil, Chile y Uruguay (2008–2022)
3. **Tasa de fertilidad: Argentina y vecinos** — Cono Sur + Bolivia y Paraguay (2014–2024)

Los datos se obtienen de la API del Banco Mundial y de Data360.

## 1. Importaciones

In [10]:
import pandas as pd
import altair as alt
import duckdb
import requests

## 2. Descarga de datos para el gráfico 1

Se usan dos fuentes del Banco Mundial:
- **WB_CSC**: Cobertura de planes sociales en el quintil más pobre (% de personas alcanzadas)
- **WB_HCP**: Gasto público en educación como % del PIB

Ambas series se filtran para Argentina y luego se unen por año.

In [2]:
# --- Cobertura de planes sociales en el quintil más pobre (Argentina) ---
url_planes = "https://data360files.worldbank.org/data360-data/data/WB_CSC/WB_CSC_PER_ALLSP_COV_Q1_TOT.csv"
duckdb.execute("CREATE OR REPLACE TABLE planes AS SELECT * FROM read_csv_auto('" + url_planes + "')")

df_planes = duckdb.sql("""
    SELECT TIME_PERIOD, REF_AREA, OBS_VALUE
    FROM planes
    WHERE REF_AREA = 'ARG'
    ORDER BY TIME_PERIOD ASC
""").to_df()

In [3]:
# --- Gasto público en educación como % del PIB (Argentina) ---
url_educacion = "https://data360files.worldbank.org/data360-data/data/WB_HCP/WB_HCP_UISXGDPFSGOV.csv"
duckdb.execute("CREATE OR REPLACE TABLE educacion AS SELECT * FROM read_csv_auto('" + url_educacion + "')")

df_educacion_arg = duckdb.sql("""
    SELECT TIME_PERIOD, REF_AREA, OBS_VALUE
    FROM educacion
    WHERE REF_AREA = 'ARG'
    ORDER BY TIME_PERIOD ASC
""").to_df()

In [4]:
# --- Unión por año (solo años presentes en ambas fuentes) ---
df_resultado = pd.merge(
    df_planes,
    df_educacion_arg,
    on='TIME_PERIOD',
    how='inner'
)[['TIME_PERIOD', 'OBS_VALUE_x', 'OBS_VALUE_y']].copy()

df_resultado.columns = ['AÑO', 'PLANES_SOCIALES', 'GASTO_EDUCACION']

# --- Asignación de gestión presidencial ---
df_resultado["Presidente"] = "CFK"
df_resultado.loc[(df_resultado["AÑO"] >= 2016) & (df_resultado["AÑO"] <= 2019), "Presidente"] = "Macri"
df_resultado.loc[df_resultado["AÑO"] >= 2020, "Presidente"] = "Fernández"

## 3. Gráfico 1 — Trayectoria: Inversión Educativa vs. Protección Social

Scatter plot con trayectoria temporal. Cada punto es un año; el color identifica la gestión presidencial.
Se muestra la etiqueta del año junto a cada punto para facilitar la lectura cronológica.

In [5]:
colores_presidentes = {
    "CFK":       "#6FA8DC",
    "Macri":     "#F1C40F",
    "Fernández": "#2ECC71"
}

# Encodings compartidos entre capas
base = alt.Chart(df_resultado).encode(
    x=alt.X(
        'GASTO_EDUCACION:Q',
        title='Gasto en Educación (% del PIB)',
        scale=alt.Scale(zero=False, padding=15)
    ),
    y=alt.Y(
        'PLANES_SOCIALES:Q',
        title='Cobertura de Planes Sociales en el Quintil más pobre (%)',
        scale=alt.Scale(zero=False, padding=15)
    ),
    order='AÑO:O'  # conecta los puntos en orden cronológico
)

# Línea de trayectoria (gris claro)
trayectoria = base.mark_line(
    color='#BEBEBE',
    strokeWidth=3,
    opacity=0.9
)

# Puntos coloreados por gestión
puntos = base.mark_circle(
    size=120,
    opacity=1,
    stroke='black',
    strokeWidth=0.5
).encode(
    color=alt.Color(
        'Presidente:N',
        scale=alt.Scale(
            domain=list(colores_presidentes.keys()),
            range=list(colores_presidentes.values())
        ),
        legend=alt.Legend(title="Gestión Presidencial", orient='bottom')
    ),
    tooltip=[
        alt.Tooltip('AÑO:O',            title='Año'),
        alt.Tooltip('Presidente:N',      title='Gestión'),
        alt.Tooltip('GASTO_EDUCACION:Q', title='Gasto Educación (% PIB)', format='.2f'),
        alt.Tooltip('PLANES_SOCIALES:Q', title='Cobertura Planes (%)',    format='.2f')
    ]
)

# Etiquetas con el año de cada punto
etiquetas = base.mark_text(
    align='left',
    dx=8, dy=-5,
    fontSize=10,
    color='#444'
).encode(text='AÑO:O')

# Composición de capas
grafico_1 = (
    trayectoria + puntos + etiquetas
).properties(
    title={
        "text": ["Trayectoria: Inversión Educativa vs. Protección Social"],
        "subtitle": [
            "Porcentaje de personas alcanzadas por planes sociales o subsidios en el quintil más pobre",
            "vs. Gasto público en educación como % del PIB. Argentina (2006–2022)"
        ],
        "fontSize": 16,
        "subtitleFontSize": 12,
        "anchor": "start"
    },
    width=700,
    height=450
).configure_axis(
    grid=False,
    domain=False,
    tickColor="#999",
    labelColor="#444"
).configure_view(
    strokeWidth=0
).interactive()

grafico_1

alt.LayerChart(...)

## 4. Descarga de datos para el gráfico 2

Se obtienen dos indicadores para cuatro países del Cono Sur (ARG, BRA, CHL, URY):
- **SI.POV.GINI**: Índice de Gini (desigualdad de ingresos)
- **SE.XPD.TOTL.GD.ZS**: Gasto público en educación como % del PIB

Fuente: API pública del Banco Mundial.

In [6]:
paises = ["ARG", "BRA", "CHL", "URY"]

# --- Índice de Gini ---
frames_gini = []
for pais in paises:
    url = f"https://api.worldbank.org/v2/country/{pais}/indicator/SI.POV.GINI?format=json&per_page=100"
    data = requests.get(url).json()
    temp = pd.DataFrame(data[1])[["countryiso3code", "date", "value"]]
    temp.columns = ["PAIS", "AÑO", "GINI"]
    frames_gini.append(temp)

df_gini = pd.concat(frames_gini)

# --- Gasto en educación (% PIB) ---
frames_edu = []
for pais in paises:
    url = f"https://api.worldbank.org/v2/country/{pais}/indicator/SE.XPD.TOTL.GD.ZS?format=json&per_page=100"
    data = requests.get(url).json()
    temp = pd.DataFrame(data[1])[["countryiso3code", "date", "value"]]
    temp.columns = ["PAIS", "AÑO", "GASTO_EDUCACION"]
    frames_edu.append(temp)

df_educacion = pd.concat(frames_edu)

# --- Unión por país y año, eliminando filas con nulos ---
df_resultado_1 = pd.merge(
    df_gini,
    df_educacion,
    on=["PAIS", "AÑO"]
).dropna()

## 5. Gráfico 2 — Relación entre inversión educativa y desigualdad

Scatter plot animado con slider de año. La trayectoria tenue muestra la evolución histórica de cada país;
el punto grande resalta la posición en el año seleccionado.

In [7]:
resultado_1 = df_resultado_1.copy()

# --- Limpieza de tipos ---
for col in ["AÑO", "GINI", "GASTO_EDUCACION"]:
    resultado_1[col] = pd.to_numeric(resultado_1[col], errors="coerce")

resultado_1 = resultado_1.dropna(subset=["AÑO", "GINI", "GASTO_EDUCACION"])

# Filtro de rango temporal para un visual más limpio
resultado_1 = resultado_1[
    (resultado_1["AÑO"] >= 2008) &
    (resultado_1["AÑO"] <= 2022)
]

colores = {
    "ARG": "#8ecae6",
    "URY": "#0A58CA",
    "BRA": "#2FBF71",
    "CHL": "#E74C3C"
}

# --- Slider interactivo de año ---
anio_slider = alt.param(
    value=2022,
    bind=alt.binding_range(min=2008, max=2022, step=1, name="Año: ")
)

base = alt.Chart(resultado_1)

# Trayectorias históricas (muy transparentes, sirven de contexto)
trayectoria = base.mark_line(
    strokeWidth=5,
    opacity=0.18
).encode(
    x=alt.X(
        "GASTO_EDUCACION:Q",
        title="Gasto educativo (% del PIB)",
        scale=alt.Scale(domain=[4, 6.5]),
        axis=alt.Axis(labelFontSize=15, titleFontSize=18, grid=False)
    ),
    y=alt.Y(
        "GINI:Q",
        title="Índice Gini",
        scale=alt.Scale(domain=[38, 56]),
        axis=alt.Axis(labelFontSize=15, titleFontSize=18, grid=False)
    ),
    color=alt.Color(
        "PAIS:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values())),
        legend=None
    ),
    detail="PAIS:N"  # línea separada por país sin necesitar color en la leyenda
)

# Punto que se mueve con el slider
puntos = base.transform_filter(
    alt.datum.AÑO == anio_slider
).mark_circle(
    size=900,
    opacity=0.95,
    stroke="white",
    strokeWidth=3
).encode(
    x="GASTO_EDUCACION:Q",
    y="GINI:Q",
    color=alt.Color(
        "PAIS:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values())),
        legend=None
    ),
    tooltip=["PAIS", "AÑO", "GINI", "GASTO_EDUCACION"]
)

# Halo blanco detrás de la etiqueta para mejorar legibilidad
texto_fondo = base.transform_filter(
    alt.datum.AÑO == anio_slider
).mark_text(
    dx=30, dy=-14,
    fontSize=16, fontWeight="bold",
    color="white", stroke="white", strokeWidth=8
).encode(x="GASTO_EDUCACION:Q", y="GINI:Q", text="PAIS:N")

# Etiqueta visible con el nombre del país
texto = base.transform_filter(
    alt.datum.AÑO == anio_slider
).mark_text(
    dx=30, dy=-14,
    fontSize=16, fontWeight="bold"
).encode(
    x="GASTO_EDUCACION:Q",
    y="GINI:Q",
    text="PAIS:N",
    color=alt.Color(
        "PAIS:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values())),
        legend=None
    )
)

# Composición final
grafico_2 = (
    trayectoria + puntos + texto_fondo + texto
).add_params(
    anio_slider
).properties(
    width=1400,
    height=820,
    title=alt.TitleParams(
        text="Relación entre inversión educativa y desigualdad",
        subtitle=["Evolución de Argentina, Brasil, Chile y Uruguay (2008–2022)"],
        anchor="start"
    )
).configure_title(
    fontSize=30,
    subtitleFontSize=18
).configure_view(
    strokeWidth=0
).interactive()

grafico_2

alt.LayerChart(...)

## 6. Descarga de datos para el gráfico 3

Se obtiene el indicador **WB_WDI_SP_DYN_TFRT_IN** (Fertility rate, total — hijos por mujer)
para Argentina y cinco vecinos del Cono Sur: Brasil, Uruguay, Chile, Bolivia y Paraguay.

Fuente: API Data360 del Banco Mundial (`/data360/data`).

In [11]:
# --- Descarga de la tasa de fertilidad desde Data360 (6 países) ---
BASE = "https://data360api.worldbank.org/data360/data"
INDICATOR = "WB_WDI_SP_DYN_TFRT_IN"
COUNTRIES = {
    "ARG": "Argentina",
    "BRA": "Brasil",
    "URY": "Uruguay",
    "CHL": "Chile",
    "BOL": "Bolivia",
    "PRY": "Paraguay",
}

frames_fertilidad = []
for code, nombre in COUNTRIES.items():
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": INDICATOR, "REF_AREA": code}
    payload = requests.get(BASE, params=params, timeout=60).json()
    rows = payload.get("value", payload) if isinstance(payload, dict) else payload
    temp = pd.DataFrame([
        {"PAIS": nombre, "AÑO": int(x["TIME_PERIOD"]), "TFR": float(x["OBS_VALUE"])}
        for x in rows
    ])
    frames_fertilidad.append(temp)

df_fertilidad = pd.concat(frames_fertilidad, ignore_index=True)

In [12]:
# --- Wrangling con DuckDB: recorte temporal y marca de Argentina como serie destacada ---
HIGHLIGHT = "Argentina"
duckdb.register("fertilidad_raw", df_fertilidad)

df_fert = duckdb.sql(f"""
    SELECT AÑO, PAIS, TFR,
           (PAIS = '{HIGHLIGHT}') AS ES_DESTACADO
    FROM fertilidad_raw
    WHERE AÑO >= 2014
    ORDER BY PAIS, AÑO
""").to_df()

## 7. Gráfico 3 — En una década, Argentina pasó de crecer a achicarse

Line chart con Argentina destacada en azul y los vecinos en gris.
Una línea de referencia horizontal marca el **nivel de reemplazo (2,1 hijos por mujer)**:
el piso a partir del cual una generación deja de reponer a la anterior.
Se anota el momento en que Argentina cruza ese umbral (entre 2017 y 2018).

In [13]:
# --- Separación de series: Argentina (destacada) vs. el resto (en gris) ---
otros = df_fert[~df_fert["ES_DESTACADO"]].copy()
arg = df_fert[df_fert["ES_DESTACADO"]].copy()

# --- Campo derivado para el tooltip: distancia al nivel de reemplazo (2,1) ---
# Permite leer de un vistazo cuán por encima o por debajo del piso de reposición
# generacional está cada observación, que es justamente lo que cuenta el gráfico.
for _df in (otros, arg):
    _df["DIF_REEMPLAZO"] = _df["TFR"] - 2.1

# --- Tooltip compartido con nombres legibles y unidades explícitas ---
tooltip_fert = [
    alt.Tooltip("PAIS:N",          title="País"),
    alt.Tooltip("AÑO:Q",           title="Año", format="d"),
    alt.Tooltip("TFR:Q",           title="Hijos por mujer",                  format=".2f"),
    alt.Tooltip("DIF_REEMPLAZO:Q", title="Diferencia con el reemplazo (2,1)", format="+.2f"),
]

# --- Escalas y ejes compartidos entre capas ---
y_scale = alt.Scale(domain=[1, 3.2], nice=False)
x_scale = alt.Scale(domain=[2014, 2024], nice=False)
x_axis = alt.Axis(labelAngle=0, values=list(range(2014, 2025, 2)), format="d")
y_axis = alt.Axis(values=[1, 2, 3], tickCount=3, format="d")

# --- Línea de referencia: nivel de reemplazo (2,1 hijos por mujer) ---
replacement = alt.Chart(pd.DataFrame({"y": [2.1]})).mark_rule(
    color="#B85C3C", strokeDash=[4, 4], strokeWidth=1
).encode(y=alt.Y("y:Q", scale=y_scale))

replacement_label = alt.Chart(
    pd.DataFrame({"y": [2.1], "x": [2014], "t": ["Nivel de reemplazo (2,1)"]})
).mark_text(
    align="left", dx=4, dy=-6, color="#B85C3C", fontSize=10
).encode(x=alt.X("x:Q", scale=x_scale), y=alt.Y("y:Q", scale=y_scale), text="t:N")

# --- Líneas de los vecinos (gris claro, transparentes) ---
otros_lineas = (
    alt.Chart(otros)
    .mark_line(strokeWidth=1.5, opacity=0.45)
    .encode(
        x=alt.X("AÑO:Q", title=None, axis=x_axis, scale=x_scale),
        y=alt.Y("TFR:Q", title=None, scale=y_scale, axis=y_axis),
        color=alt.value("#bdbdbd"),
        detail="PAIS:N",
    )
)

# --- Puntos invisibles sobre cada observación para enganchar el tooltip ---
otros_puntos = (
    alt.Chart(otros)
    .mark_point(size=60, opacity=0)
    .encode(
        x=alt.X("AÑO:Q", scale=x_scale),
        y=alt.Y("TFR:Q", scale=y_scale),
        tooltip=tooltip_fert,
    )
)

# --- Etiquetas finales con offset manual para evitar superposición ---
offsets_etiquetas = {
    "Bolivia": 2.55, "Paraguay": 2.38, "Brasil": 1.65,
    "Argentina": 1.50, "Uruguay": 1.35, "Chile": 1.14,
}
ultimo_anio = int(df_fert["AÑO"].max())
df_etiquetas = pd.DataFrame([
    {"AÑO": ultimo_anio, "PAIS": p, "y": y, "ES_DESTACADO": p == HIGHLIGHT}
    for p, y in offsets_etiquetas.items()
])

otros_etiquetas = (
    alt.Chart(df_etiquetas[~df_etiquetas["ES_DESTACADO"]])
    .mark_text(align="left", dx=6, fontSize=10, color="#888")
    .encode(x=alt.X("AÑO:Q", scale=x_scale), y=alt.Y("y:Q", scale=y_scale), text="PAIS:N")
)

# --- Línea de Argentina (destacada en azul, más gruesa) ---
arg_linea = (
    alt.Chart(arg)
    .mark_line(strokeWidth=3, color="#3CA5DC")
    .encode(
        x=alt.X("AÑO:Q", scale=x_scale),
        y=alt.Y("TFR:Q", scale=y_scale),
    )
)

# --- Puntos invisibles sobre Argentina para enganchar el tooltip año a año ---
arg_puntos = (
    alt.Chart(arg)
    .mark_point(size=80, opacity=0)
    .encode(
        x=alt.X("AÑO:Q", scale=x_scale),
        y=alt.Y("TFR:Q", scale=y_scale),
        tooltip=tooltip_fert,
    )
)

arg_etiqueta = (
    alt.Chart(df_etiquetas[df_etiquetas["ES_DESTACADO"]])
    .mark_text(align="left", dx=6, fontSize=11, fontWeight="bold", color="#3CA5DC")
    .encode(x=alt.X("AÑO:Q", scale=x_scale), y=alt.Y("y:Q", scale=y_scale), text="PAIS:N")
)

# --- Anotación: Argentina cruza el reemplazo entre 2017 (2,168) y 2018 (2,067) ---
# Interpolación lineal sobre y=2,1 -> x ≈ 2017,67
cross_x, cross_y = 2017.67, 2.1
df_cruce = pd.DataFrame({"AÑO": [cross_x], "TFR": [cross_y]})
punto_cruce = (
    alt.Chart(df_cruce)
    .mark_point(size=90, filled=True, color="#3CA5DC", stroke="white", strokeWidth=2)
    .encode(x=alt.X("AÑO:Q", scale=x_scale), y=alt.Y("TFR:Q", scale=y_scale))
)
df_anotacion = pd.DataFrame({
    "AÑO": [2015.6], "TFR": [1.97],
    "t": ["Argentina cruza\nel nivel de reemplazo"],
})
texto_cruce = (
    alt.Chart(df_anotacion)
    .mark_text(align="left", baseline="top", fontSize=10, color="#333", lineBreak="\n")
    .encode(x=alt.X("AÑO:Q", scale=x_scale), y=alt.Y("TFR:Q", scale=y_scale), text="t:N")
)
df_flecha = pd.DataFrame({"x": [cross_x], "x2": [2016.8], "y": [cross_y], "y2": [1.97]})
flecha_cruce = (
    alt.Chart(df_flecha)
    .mark_rule(color="#666", strokeWidth=1)
    .encode(
        x=alt.X("x:Q", scale=x_scale), x2="x2:Q",
        y=alt.Y("y:Q", scale=y_scale), y2="y2:Q",
    )
)

# --- Composición final ---
grafico_3 = (
    replacement + replacement_label
    + otros_lineas + otros_puntos + otros_etiquetas
    + arg_linea + arg_puntos + arg_etiqueta
    + flecha_cruce + punto_cruce + texto_cruce
).properties(
    width=820,
    height=380,
    title=alt.TitleParams(
        "En una década, Argentina pasó de crecer a achicarse",
        subtitle=[
            "Su tasa de fertilidad atravesó en 2018 el nivel de reemplazo (2,1 hijos por mujer, el piso para que cada generación reponga a la anterior) y siguió cayendo hasta 1,4.",
            "Toda la región cae al mismo tiempo: Brasil, Chile y Uruguay ya estaban debajo, Bolivia y Paraguay vienen desde más arriba.",
        ],
        anchor="start",
        fontSize=18,
        subtitleFontSize=12,
        subtitleColor="#555",
        subtitlePadding=6,
        offset=10,
    ),
    padding={"left": 10, "top": 10, "right": 80, "bottom": 10},
).configure_view(
    strokeOpacity=0
).configure_axis(
    grid=False, domainColor="#ccc", tickColor="#ccc",
    labelColor="#444", titleColor="#444",
)

grafico_3

alt.LayerChart(...)

In [ ]:

# --- Exportar los tres gráficos a JSON para incrustar en GitHub Pages ---
from pathlib import Path

out = Path(".")
grafico_1.save(str(out / "grafico_1.json"))
grafico_2.save(str(out / "grafico_2.json"))
grafico_3.save(str(out / "grafico_3.json"))
print("JSONs guardados:", [str(p) for p in out.glob("grafico_*.json")])
